# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [ ]:
import sys
import os
import re
import json
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import re as _re

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider


import torch
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold

from openai import OpenAI

from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider


# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # 4B alternative
# EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")
import time
import platform

def get_device_info():
    """Return device info dict."""
    import torch
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }
import matplotlib.pyplot as plt
from dotenv import load_dotenv


- Why `BAAI/bge-base-en-v1.5`?
  - Strong MTEB performance. On the [Massive Text Embedding Benchmark](https://huggingface.co/spaces/mteb/leaderboard), bge-base is among the top-performing models in its size class (approximately 110M parameters).
  - Cross-modal capability for code and natural language. The bge series performs well on tasks such as code search and code–natural language matching, making it suitable for measuring cross-modal similarity between natural-language CVE descriptions and PDDL code.

## 2. Load data, Prompts, LLM (tokenizer and embedding model)

In [2]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...
Generated domain dir to be evaluated: ../../generated_domain/eval_set


In [4]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model.
    Qwen3-Embedding models require trust_remote_code=True.
    """
    if "Qwen3-Embedding" in model_name:
        return SentenceTransformer(model_name, trust_remote_code=True)
    return SentenceTransformer(model_name)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer

TARGET_POOL_TI_FILE = os.path.join(PROJECT_ROOT, "resources", "data", "target_pool_ti.json")

def load_target_pool_ti(path):
    """Load TI-enriched target pool. Returns {cve_id: {field: value}}."""
    with open(path, encoding="utf-8") as f:
        pool = json.load(f)
    return {entry["cve_id"]: entry for entry in pool}

def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
prompt_env = load_prompts(PROMPTS_PATH)

ti_pool = load_target_pool_ti(TARGET_POOL_TI_FILE)
print(f"TI pool loaded: {len(ti_pool)} CVEs")

embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    import random as _rng
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Save LLM-as-expert results ---
# RESULTS_BASE defined in Cell 2

def save_llm_results(results, eval_type, mode, model_name, elapsed_total=None):
    """Save LLM-as-expert evaluation results to JSON.
    
    Args:
        results: list of result dicts [{cve_id, ap_id, ...}]
        eval_type: 'intrinsic' or 'extrinsic'
        mode: 'binary' or 'scored'
        elapsed_total: total wall-clock seconds (optional)
    """
    import platform
    
    safe_model = model_name.replace("/", "_").replace(":", "_")
    out_dir = Path(RESULTS_BASE) / "semantic" / eval_type / "llm-as-experts"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    results_path = out_dir / f"results_{mode}_{safe_model}.json"
    metadata_path = out_dir / f"metadata_{mode}_{safe_model}.json"
    
    # Results
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    # Metadata
    total_prompt = sum(r.get("usage", {}).get("prompt_tokens", 0) for r in results)
    total_completion = sum(r.get("usage", {}).get("completion_tokens", 0) for r in results)
    metadata = {
        "eval_type": eval_type,
        "mode": mode,
        "model": model_name,
        "n_domains": len(results),
        "total_prompt_tokens": total_prompt,
        "total_completion_tokens": total_completion,
        "total_tokens": total_prompt + total_completion,
        "time_seconds": elapsed_total,
        "device_info": {
            "platform": platform.platform(),
            "python": platform.python_version(),
        },
    }
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"Saved {len(results)} results to {results_path}")
    print(f"Saved metadata to {metadata_path}")


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---
from sklearn.metrics import classification_report

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...
Generated domain dir to be evaluated: ../../generated_domain/eval_set
TI pool loaded: 26 CVEs


Device set to use cpu


## 3. Evaluation

### 3.1 Syntax Check (ENHSP)

In [65]:
enhsp = create_enhsp_checker()

t_start_syntax = time.time()
reference_syntax_results = []
for entry in dataset:
    for ap in entry["attack_paths"]:
        problem_str = generate_problem(ap["domain"])
        r_enhsp = enhsp.check_from_string(ap["domain"], problem_str)
        result = {
            "cve_id": entry["cve_id"],
            "ap_id": ap["ap_id"],
            "syntax_ok": r_enhsp.success,
            "error": r_enhsp.error,
        }
        reference_syntax_results.append(result)
        print(f"{entry['cve_id']}/{ap['ap_id']}  syntax: {r_enhsp.success}")

t_syntax = time.time() - t_start_syntax
n_pass = sum(1 for r in reference_syntax_results if r["syntax_ok"])
print(f"\nSyntax: {n_pass}/{len(reference_syntax_results)} passed, time: {t_syntax:.2f}s")

# Save
save_dir = os.path.join(RESULTS_BASE, "syntax")
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, "results.json"), "w") as f:
    json.dump(reference_syntax_results, f, indent=2)
with open(os.path.join(save_dir, "metadata.json"), "w") as f:
    json.dump({"time_seconds": t_syntax, "n_domains": len(reference_syntax_results),
               "tool": "ENHSP", "device_info": get_device_info()}, f, indent=2)
print(f"Saved to {save_dir}")


Python(36378) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2022-1471/AP1  syntax: True


Python(36383) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2022-40149/AP1  syntax: True


Python(36417) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

CVE-2023-44487/AP1  syntax: True


Python(36621) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2023-46589/AP1  syntax: True


Python(36683) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


KeyboardInterrupt: 

### 3.2 Solvability (Metric-FF)

In [66]:
ff = create_ff_checker()

t_start_solv = time.time()
reference_solvability_results = []
for entry in dataset:
    for ap in entry["attack_paths"]:
        problem_str = generate_problem(ap["domain"])
        r_ff = ff.check_from_string(ap["domain"], problem_str)
        result = {
            "cve_id": entry["cve_id"],
            "ap_id": ap["ap_id"],
            "solvable": r_ff.solvable,
            "plan_length": r_ff.plan_length,
            "plan": r_ff.plan,
            "error": r_ff.error,
        }
        reference_solvability_results.append(result)
        status = "SOLVABLE" if r_ff.solvable else "FAIL"
        print(f"{entry['cve_id']}/{ap['ap_id']}  {status}  plan_length={r_ff.plan_length}")
        if r_ff.plan:
            for i, action in enumerate(r_ff.plan):
                print(f"  {i}: {action}")

t_solv = time.time() - t_start_solv
n_solvable = sum(1 for r in reference_solvability_results if r["solvable"])
print(f"\nSolvability: {n_solvable}/{len(reference_solvability_results)} solvable, time: {t_solv:.2f}s")

# Save
save_dir = os.path.join(RESULTS_BASE, "solvability")
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, "results.json"), "w") as f:
    json.dump(reference_solvability_results, f, indent=2)
with open(os.path.join(save_dir, "metadata.json"), "w") as f:
    json.dump({"time_seconds": t_solv, "n_domains": len(reference_solvability_results),
               "tool": "Metric-FF", "device_info": get_device_info()}, f, indent=2)
print(f"Saved to {save_dir}")


Python(37104) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37105) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37106) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37107) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2022-1471/AP1  SOLVABLE  plan_length=10
  0: ATTACKER-CRAFTS-MALICIOUS-JAVA-CLASS SEFA JAVA-GADGET-CLASS_SEFA COMMAND-EXECUTION-LOGIC_SEFA CVE_2022_1471
  1: ATTACKER-CRAFTS-MALICIOUS-YAML-PAYLOAD SEFA YAML-PAYLOAD_SEFA JAVA-GADGET-CLASS_SEFA JNDI-ENDPOINT_SEFA JAVA-GADGET-CLASS_SEFA CVE_2022_1471
  2: ATTACKER-SENDS-MALICIOUS-YAML-VIA-HTTP-POST HTTP-POST-REQUEST_SEFA SEFA YAML-PAYLOAD_SEFA YAML-ENDPOINT_SEFA CVE_2022_1471
  3: TARGET-SYSTEM-RECEIVES-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA YAML-PAYLOAD_SEFA YAML-ENDPOINT_SEFA
  4: TARGET-SYSTEM-PASSES-YAML-TO-SNAKEYAML-LOADER HTTP-POST-REQUEST_SEFA YAML-PAYLOAD_SEFA SEFA SNAKEYAML-LIBRARY_SEFA
  5: TARGET-SYSTEM-STARTS-YAML-DESERIALIZATION-WITH-CONSTRUCTOR YAML-PAYLOAD_SEFA SNAKEYAML-LIBRARY_SEFA YAML-CONSTRUCTOR_SEFA SEFA
  6: TARGET-SYSTEM-INSTANTIATES-DANGEROUS-JAVA-GADGET-CLASS SEFA YAML-CONSTRUCTOR_SEFA JAVA-GADGET-CLASS_SEFA YAML-PAYLOAD_SEFA
  7: TARGET-SYSTEM-LOADS-MALICIOUS-CLASS-FROM-JNDI-ENDP

Python(37108) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37109) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37110) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37111) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2022-40150/AP2  SOLVABLE  plan_length=9
  0: ATTACKER-CRAFTS-MALICIOUS-XML-PAYLOAD SEFA XML-PAYLOAD_SEFA JAVA-LIBRARY_SEFA CVE_2022_40150
  1: ATTACKER-SENDS-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA XML-PAYLOAD_SEFA SOAP-ENDPOINT_SEFA CVE_2022_40150
  2: TARGET-SYSTEM-RECEIVES-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA XML-PAYLOAD_SEFA SOAP-ENDPOINT_SEFA
  3: TARGET-SYSTEM-EXTRACTS-XML-PAYLOAD-FROM-HTTP-REQUEST SEFA HTTP-POST-REQUEST_SEFA XML-PAYLOAD_SEFA SOAP-ENDPOINT_SEFA
  4: TARGET-SYSTEM-INVOKES-JETTISON-XML-TO-JSON-TRANSFORMATION SEFA JAVA-LIBRARY_SEFA XML-PAYLOAD_SEFA JSON-PAYLOAD_SEFA
  5: TARGET-SYSTEM-USES-JETTISON-JSON-PARSER SEFA JAVA-LIBRARY_SEFA JSON-PAYLOAD_SEFA
  6: TARGET-SYSTEM-JETTISON-PARSER-ALLOCATES-HEAP-MEMORY-FOR-JSON-CONSTRUCTION JAVA-LIBRARY_SEFA JSON-PAYLOAD_SEFA SEFA
  7: TARGET-SYSTEM-TRIGGERS-OUTOFMEMORY-ERROR SEFA CVE_2022_40150
  8: TARGET-SYSTEM-BECOMES-UNRESPONSIVE SEFA
CVE-2023-2976/AP1  SOLVAB

Python(37112) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37113) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37114) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37115) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

Python(37148) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37149) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37150) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37151) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37152) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2024-47072/AP1  SOLVABLE  plan_length=8
  0: ATTACKER-CRAFTS-MALICIOUS-BINARY-XSTREAM-PAYLOAD SEFA BINARY-PAYLOAD_SEFA JAVA-LIBRARY_SEFA CVE_2024_47072
  1: ATTACKER-SENDS-HTTP-POST-REQUEST-WITH-BINARY-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA BINARY-PAYLOAD_SEFA API-ENDPOINT_SEFA CVE_2024_47072
  2: TARGET-SYSTEM-RECEIVES-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA BINARY-PAYLOAD_SEFA API-ENDPOINT_SEFA
  3: TARGET-SYSTEM-EXTRACTS-BINARY-PAYLOAD-FROM-HTTP-REQUEST SEFA HTTP-POST-REQUEST_SEFA BINARY-PAYLOAD_SEFA API-ENDPOINT_SEFA
  4: TARGET-SYSTEM-USES-XSTREAM-BINARY-DRIVER SEFA JAVA-LIBRARY_SEFA BINARY-STREAM-DRIVER_SEFA BINARY-PAYLOAD_SEFA
  5: TARGET-SYSTEM-BINARY-DRIVER-ALLOCATES-STACK-FRAMES-FOR-RECURSIVE-DECODING JAVA-LIBRARY_SEFA BINARY-STREAM-DRIVER_SEFA BINARY-PAYLOAD_SEFA SEFA
  6: TARGET-SYSTEM-TRIGGERS-STACKOVERFLOW-ERROR SEFA CVE_2024_47072
  7: TARGET-SYSTEM-BECOMES-UNRESPONSIVE SEFA
CVE-2025-22228/AP1  SOLVABLE  plan_length=9
  0: TARGET-SYSTEM-EXPO

Python(37153) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37154) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37155) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37156) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(37157) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CVE-2025-24813/AP2  SOLVABLE  plan_length=9
  0: ATTACKER-CREATES-MALICIOUS-CONTENT-FOR-INJECTION SEFA MALICIOUS-CONTENT_SEFA SENSITIVE-FILE_SEFA CVE_2025_24813
  1: ATTACKER-CRAFTS-HTTP-PUT-REQUEST-WITH-INTERNAL-DOT-PATH-MANIPULATION HTTP-PUT-REQUEST_SEFA SENSITIVE-FILE_SEFA PUBLIC-DIRECTORY_SEFA SENSITIVE-DIRECTORY_SEFA SEFA CVE_2025_24813 MALICIOUS-CONTENT_SEFA
  2: ATTACKER-INCLUDES-CONTENT-RANGE-HEADERS-FOR-PARTIAL-PUT HTTP-PUT-REQUEST_SEFA CONTENT-RANGE-HEADERS_SEFA SEFA CVE_2025_24813
  3: ATTACKER-SENDS-MALICIOUS-PARTIAL-HTTP-PUT-REQUEST-TO-TOMCAT HTTP-PUT-REQUEST_SEFA CONTENT-RANGE-HEADERS_SEFA SEFA TOMCAT_SEFA MALICIOUS-CONTENT_SEFA CVE_2025_24813
  4: TARGET-SYSTEM-RECEIVES-HTTP-PUT-REQUEST HTTP-PUT-REQUEST_SEFA SEFA
  5: TARGET-SYSTEM-TOMCAT-DEFAULT-SERVLET-PROCESSES-REQUEST-WITHOUT-PROPER-VALIDATION HTTP-PUT-REQUEST_SEFA SEFA DEFAULT-SERVLET_SEFA TOMCAT_SEFA
  6: TARGET-SYSTEM-TOMCAT-DEFAULT-SERVLET-INCORRECTLY-NORMALIZES-PATH HTTP-PUT-REQUEST_SEFA DEFAULT-SERVLET_SEFA TOM

Python(37158) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


### 3.3 Semantic Evaluation

#### 3.3.1 Intrinsic

##### 3.3.1.1a Embedding: NL CVE Description vs PDDL Similarity

Block 1:  the function to compute the embedding similarity between the NL CVE description vs the PDDL code (domain)

In [5]:
def embedding_similarity_intrinsic(descriptions, pddl_texts, model):
    """Cosine similarity matrix between NL descriptions and PDDL codes.
    Args:
        descriptions: list of CVE NL description strings
        pddl_texts: list of PDDL (domain) strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(descriptions), len(pddl_texts))
    """
    E_desc = model.encode(
        descriptions,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_pddl = model.encode(
        pddl_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_desc @ E_pddl.T
    return S.float().cpu().numpy()

Block 2: Cross-validation, performance estimation, and corrected proportion

Process:
1. Build positive/negative pairs from reference data 
   - 1:PDDL doman vs same CVE NL desc 
   - 0: PDDL doman vs different CVE NL desc
2. CVE-level GroupKFold CV with bootstrap threshold on each train fold
3. Out-of-fold predictions → global confusion matrix → TPR, FPR, classification report
4. Apply calibrated threshold to generated domains → corrected proportion π = (PPV - FPR) / (TPR - FPR)

Why GroupKFold?
- one CVE NL desc corresponded to multiple PDDL AP domain 
- If the train fold contains (for example, CVE-2024-12798 NL desc vs domain_AP4) and the validation fold contains (CVE-2024-12798 NL desc vs domain_AP1), the threshold is calibrated on an embedding the validation set also shares, this is data leakage. GroupKFold ensures all pairs involving the same CVE NL desc are assigned to the same fold, eliminating this issue.

In [6]:
# Step 1: Build positive/negative pairs

def build_intrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for intrinsic embedding evaluation.
    Positive (1): CVE domain vs same CVE NL description
    Negative (0): CVE domain vs different CVE NL description
    Groups: assigned by the description-side CVE
    Reuses embedding_similarity_intrinsic for batch computation.
    """
    descs, domains, cve_ids = [], [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            descs.append(entry["description"])
            domains.append(ap["domain"])
            cve_ids.append(entry["cve_id"])

    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)

    scores, labels, groups = [], [], []
    n = len(descs)
    for i in range(n):
        for j in range(n):
            scores.append(float(sim_matrix[i, j]))
            labels.append(1 if cve_ids[i] == cve_ids[j] else 0)
            groups.append(cve_ids[i])

    return np.array(scores), np.array(labels), np.array(groups)


pair_scores_a, pair_labels_a, pair_groups_a = build_intrinsic_pairs(dataset, embedding_model)
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {pair_labels_a.sum()}, Negative pairs: {(pair_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(pair_groups_a))}")

# Global list for cross-model comparison CSV
calibration_embedding_rows = []


Embedding Model: all-MiniLM-L6-v2
  Positive pairs: 203, Negative pairs: 2822
  Groups (CVEs): 21


In [7]:
# Step 2: CV calibration
def _find_threshold_pr(y_true, y_score):
    """Return the threshold closest to (1,1) in the precision-recall curve."""
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    distances = np.sqrt((1 - prec[1:]) ** 2 + (1 - rec[1:]) ** 2)
    return float(thr[np.argmin(distances)])

def run_calibration(scores, labels, groups, k=5, n_bootstrap=500, random_state=42):
    """
    CVE-level GroupKFold CV with bootstrap threshold search on each train fold.
    Returns: median_threshold, fold_thresholds, y_pred (OOF), y_true (OOF)
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups)
    rng = np.random.RandomState(random_state)

    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)
    actual_k = min(k, n_groups)
    if actual_k < k:
        print(f"Warning: only {n_groups} groups, reducing k from {k} to {actual_k}")

    gkf = GroupKFold(n_splits=actual_k)
    fold_thresholds, y_pred_parts, y_true_parts = [], [], []

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(scores, labels, groups)):
        tr_scores, tr_labels = scores[train_idx], labels[train_idx]
        bt = []
        for _ in range(n_bootstrap):
            idx = rng.choice(len(tr_scores), size=len(tr_scores), replace=True)
            bt.append(_find_threshold_pr(tr_labels[idx], tr_scores[idx]))
        fold_thr = float(np.median(bt))
        fold_thresholds.append(fold_thr)
        y_pred_parts.append((scores[val_idx] >= fold_thr).astype(int))
        y_true_parts.append(labels[val_idx])
        val_groups = np.unique(groups[val_idx])
        print(f"  Fold {fold_i+1}: threshold={fold_thr:.4f}, val CVEs={list(val_groups)}")

    return (
        float(np.median(fold_thresholds)),
        fold_thresholds,
        np.concatenate(y_pred_parts),
        np.concatenate(y_true_parts),
    )


cv_threshold_a, cv_fold_thr_a, cv_pred_a, cv_true_a = run_calibration(
    pair_scores_a, pair_labels_a, pair_groups_a)
print("Fold thresholds:", [f"{t:.4f}" for t in cv_fold_thr_a])
print(f"Median threshold: {cv_threshold_a:.4f}")


  Fold 1: threshold=0.3730, val CVEs=['CVE-2023-44487', 'CVE-2024-12798', 'CVE-2024-34447', 'CVE-2024-38809']
  Fold 2: threshold=0.4499, val CVEs=['CVE-2023-33202', 'CVE-2023-34055', 'CVE-2024-22243', 'CVE-2024-38816']
  Fold 3: threshold=0.4039, val CVEs=['CVE-2022-40150', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2025-22228']
  Fold 4: threshold=0.4596, val CVEs=['CVE-2022-40149', 'CVE-2024-22259', 'CVE-2024-47072', 'CVE-2025-24813']
  Fold 5: threshold=0.4005, val CVEs=['CVE-2022-1471', 'CVE-2023-2976', 'CVE-2023-46589', 'CVE-2023-6378', 'CVE-2024-38286']
Fold thresholds: ['0.3730', '0.4499', '0.4039', '0.4596', '0.4005']
Median threshold: 0.4039


In [8]:
# Step 3: Performance report

def report_row(y_true, y_pred, **meta):
    """Return a flat dict with classification metrics + TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f"{key}__{metric}"] = v
        else:
            row[key] = val
    row["tpr"] = rpt.get("1", {}).get("recall", float("nan"))
    row["fpr"] = 1.0 - rpt.get("0", {}).get("recall", float("nan"))
    return row


print(f"Embedding Model ({EMBEDDING_MODEL_NAME})")
print(classification_report(cv_true_a, cv_pred_a, zero_division=0))

row_a = report_row(cv_true_a, cv_pred_a,
                   metric="embedding", model=EMBEDDING_MODEL_NAME,
                   mode="intrinsic", threshold=cv_threshold_a)
print(f"TPR = {row_a['tpr']:.4f}  FPR = {row_a['fpr']:.4f}")


# Save to CSV (Marco's pattern: one row per embedding model)
import platform, pandas as pd

row_a["time_seconds"] = t_elapsed_a if 't_elapsed_a' in dir() else None
row_a["device"] = "GPU" if torch.cuda.is_available() else "CPU"
row_a["platform"] = platform.platform()

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
csv_path = os.path.join(save_dir, "intrinsic_similarity.csv")

# Append or create
if os.path.exists(csv_path):
    existing = pd.read_csv(csv_path)
    # Remove old row for same model if exists
    existing = existing[existing.get("model", "") != EMBEDDING_MODEL_NAME]
    df = pd.concat([existing, pd.DataFrame([row_a])], ignore_index=True)
else:
    df = pd.DataFrame([row_a])

df.to_csv(csv_path, index=False)
print(df[["model", "threshold", "tpr", "fpr", "accuracy", "time_seconds", "device"]].to_string(index=False))


Embedding Model (all-MiniLM-L6-v2)
              precision    recall  f1-score   support

           0       0.97      0.87      0.91      2822
           1       0.24      0.56      0.33       203

    accuracy                           0.85      3025
   macro avg       0.60      0.72      0.62      3025
weighted avg       0.92      0.85      0.88      3025

TPR = 0.5616  FPR = 0.1304
                  model  threshold      tpr      fpr  accuracy time_seconds device
Qwen/Qwen3-Embedding-4B   0.733721 0.463054 0.047838  0.919339          NaN    CPU
Qwen/Qwen3-Embedding-8B   0.733721 0.463054 0.047838  0.919339          NaN    CPU
       all-MiniLM-L6-v2   0.403923 0.561576 0.130404  0.848926         None    CPU


In [10]:
# Step 4: Apply threshold to reference domains + corrected proportion π 

def apply_threshold_batch(descriptions, domains, labels_list, model, threshold):
    """Apply threshold using batch embedding_similarity_intrinsic. Returns results and PPV."""
    sim_matrix = embedding_similarity_intrinsic(descriptions, domains, model)
    results = []
    for i, label in enumerate(labels_list):
        sim = float(sim_matrix[i, i])
        pred = 1 if sim >= threshold else 0
        results.append({"label": label, "similarity": sim, "prediction": pred})
        print(f"  {label:45s} prediction: {pred} (similarity: {sim:.4f})")
    predictions = [r["prediction"] for r in results]
    ppv = sum(predictions) / len(predictions) if predictions else 0
    return results, ppv


def corrected_proportion(ppv, tpr, fpr):
    """Corrected estimate of true positive proportion: pi = (PPV - FPR) / (TPR - FPR)"""
    denom = tpr - fpr
    if abs(denom) < 1e-10:
        return float("nan")
    return (ppv - fpr) / denom


# ── Apply to reference domains ──
ref_descs, ref_domains, ref_labels = [], [], []
for entry in dataset:
    for ap in entry["attack_paths"]:
        ref_descs.append(entry["description"])
        ref_domains.append(ap["domain"])
        ref_labels.append(f"{entry['cve_id']}/{ap['ap_id']}")

print(f"Applying threshold {cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to reference domains:")
ref_results_a, ref_ppv_a = apply_threshold_batch(ref_descs, ref_domains, ref_labels, embedding_model, cv_threshold_a)
print(f"  Reference PPV: {ref_ppv_a:.4f} ({sum(r['prediction'] for r in ref_results_a)}/{len(ref_results_a)} predicted positive)")



# ── Save (reference only for now) ──
save_data = {
    "metric": "embedding",
    "model": EMBEDDING_MODEL_NAME,
    "mode": "intrinsic",
    "median_threshold": cv_threshold_a,
    "fold_thresholds": cv_fold_thr_a,
    "tpr": row_a["tpr"],
    "fpr": row_a["fpr"],
    "classification_report": classification_report(cv_true_a, cv_pred_a, output_dict=True, zero_division=0),
    "n_positive_pairs": int(pair_labels_a.sum()),
    "n_negative_pairs": int((pair_labels_a == 0).sum()),
    "n_groups": int(len(np.unique(pair_groups_a))),
    "ref_ppv": ref_ppv_a,
    "reference_predictions": ref_results_a,
}
save_path = Path(EVAL_SET_DIR) / f"cv_calibration_{EMBEDDING_MODEL_NAME.replace('/', '_')}.json"
save_path.write_text(json.dumps(save_data, indent=2), encoding="utf-8")

# Append to global calibration rows
calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME,
    "mode": "intrinsic",
    "threshold": cv_threshold_a,
    "tpr": row_a["tpr"],
    "fpr": row_a["fpr"],
    "1__precision": row_a.get("1__precision", float("nan")),
    "1__recall": row_a.get("1__recall", float("nan")),
    "1__f1-score": row_a.get("1__f1-score", float("nan")),
    "accuracy": row_a.get("accuracy", float("nan")),
    "ref_ppv": ref_ppv_a,
    "n_positive": int(pair_labels_a.sum()),
    "n_negative": int((pair_labels_a == 0).sum()),
})


Applying threshold 0.4039 (all-MiniLM-L6-v2) to reference domains:
  CVE-2022-1471/AP1                             prediction: 0 (similarity: 0.3815)
  CVE-2022-40149/AP1                            prediction: 0 (similarity: 0.3739)
  CVE-2022-40149/AP2                            prediction: 0 (similarity: 0.3739)
  CVE-2022-40150/AP1                            prediction: 0 (similarity: 0.3650)
  CVE-2022-40150/AP2                            prediction: 0 (similarity: 0.3650)
  CVE-2023-2976/AP1                             prediction: 0 (similarity: 0.3337)
  CVE-2023-2976/AP2                             prediction: 0 (similarity: 0.3296)
  CVE-2023-2976/AP3                             prediction: 0 (similarity: 0.3249)
  CVE-2023-33202/AP1                            prediction: 0 (similarity: 0.3758)
  CVE-2023-33202/AP2                            prediction: 0 (similarity: 0.3848)
  CVE-2023-33202/AP3                            prediction: 0 (similarity: 0.3822)
  CVE-2023-34055/AP1

##### 3.3.1.1e Embedding: TI-Selected NL vs Action Names

Use current embedding model with **TI-selected NL** (description + CAPEC + CVSS vector) on the NL side, and **extracted action names only** (no preconditions/effects) on the PDDL side.

Same 4-step structure as 3.3.1.1a: build pairs → CV calibration → performance report → apply threshold.


In [11]:
# Step 1: Build positive/negative pairs (TI-selected NL vs action names)

def build_intrinsic_pairs_ti_action(dataset, ti_pool, model):
    """
    Build (scores, labels, groups) for TI-selected NL vs action names.
    Positive (1): TI-NL vs action names of same CVE
    Negative (0): TI-NL vs action names of different CVE
    """
    descs, action_texts, cve_ids = [], [], []
    for entry in dataset:
        ti_entry = ti_pool.get(entry["cve_id"])
        if ti_entry is None:
            continue
        nl_ti = build_nl_ti_selected(ti_entry)
        for ap in entry["attack_paths"]:
            action_names = re.findall(r"\(:action\s+([\w-]+)", ap["domain"])
            descs.append(nl_ti)
            action_texts.append(" ".join(action_names))
            cve_ids.append(entry["cve_id"])

    sim_matrix = embedding_similarity_intrinsic(descs, action_texts, model)

    scores, labels, groups = [], [], []
    n = len(descs)
    for i in range(n):
        for j in range(n):
            scores.append(float(sim_matrix[i, j]))
            labels.append(1 if cve_ids[i] == cve_ids[j] else 0)
            groups.append(cve_ids[i])

    return np.array(scores), np.array(labels), np.array(groups)


t_start_e = time.time()
pair_scores_e, pair_labels_e, pair_groups_e = build_intrinsic_pairs_ti_action(dataset, ti_pool, embedding_model)
t_elapsed_e = time.time() - t_start_e
print(f"Embedding Model: {EMBEDDING_MODEL_NAME} (TI-selected NL vs action names)")
print(f"  Positive pairs: {pair_labels_e.sum()}, Negative pairs: {(pair_labels_e == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(pair_groups_e))}")
print(f"  Time: {t_elapsed_e:.2f}s")


Embedding Model: all-MiniLM-L6-v2 (TI-selected NL vs action names)
  Positive pairs: 203, Negative pairs: 2822
  Groups (CVEs): 21
  Time: 4.12s


In [12]:
# Step 2: CV calibration (reuses run_calibration from 3.3.1.1a)

cv_threshold_e, cv_fold_thr_e, cv_pred_e, cv_true_e = run_calibration(
    pair_scores_e, pair_labels_e, pair_groups_e)
print("Fold thresholds:", [f"{t:.4f}" for t in cv_fold_thr_e])
print(f"Median threshold: {cv_threshold_e:.4f}")


  Fold 1: threshold=0.5534, val CVEs=['CVE-2023-44487', 'CVE-2024-12798', 'CVE-2024-34447', 'CVE-2024-38809']
  Fold 2: threshold=0.5002, val CVEs=['CVE-2023-33202', 'CVE-2023-34055', 'CVE-2024-22243', 'CVE-2024-38816']
  Fold 3: threshold=0.5410, val CVEs=['CVE-2022-40150', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2025-22228']
  Fold 4: threshold=0.4992, val CVEs=['CVE-2022-40149', 'CVE-2024-22259', 'CVE-2024-47072', 'CVE-2025-24813']
  Fold 5: threshold=0.5002, val CVEs=['CVE-2022-1471', 'CVE-2023-2976', 'CVE-2023-46589', 'CVE-2023-6378', 'CVE-2024-38286']
Fold thresholds: ['0.5534', '0.5002', '0.5410', '0.4992', '0.5002']
Median threshold: 0.5002


In [14]:
# Step 3: Performance report + save

print(f"Embedding Model ({EMBEDDING_MODEL_NAME}, TI-selected NL vs action names)")
print(classification_report(cv_true_e, cv_pred_e, zero_division=0))

row_e = report_row(cv_true_e, cv_pred_e,
                   metric="embedding", model=EMBEDDING_MODEL_NAME,
                   mode="intrinsic", variant="ti_action_names",
                   threshold=cv_threshold_e)
print(f"TPR = {row_e['tpr']:.4f}  FPR = {row_e['fpr']:.4f}")

# Save to CSV
import platform, pandas as pd

row_e["time_seconds"] = t_elapsed_e
row_e["device"] = "GPU" if torch.cuda.is_available() else "CPU"
row_e["platform"] = platform.platform()

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
csv_path = os.path.join(save_dir, "intrinsic_similarity.csv")

if os.path.exists(csv_path):
    existing = pd.read_csv(csv_path)
    # Remove old row for same model+variant
    mask = ~((existing.get("model", pd.Series()) == EMBEDDING_MODEL_NAME) & 
             (existing.get("variant", pd.Series()) == "ti_action_names"))
    existing = existing[mask]
    df = pd.concat([existing, pd.DataFrame([row_e])], ignore_index=True)
else:
    df = pd.DataFrame([row_e])

df.to_csv(csv_path, index=False)
calibration_embedding_rows.append(row_e)


Embedding Model (all-MiniLM-L6-v2, TI-selected NL vs action names)
              precision    recall  f1-score   support

           0       0.97      0.88      0.92      2822
           1       0.26      0.59      0.36       203

    accuracy                           0.86      3025
   macro avg       0.61      0.73      0.64      3025
weighted avg       0.92      0.86      0.88      3025

TPR = 0.5862  FPR = 0.1187


##### 3.3.1.2 LLM as an Expert: NL CVE Description vs PDDL Match

##### 3.3.1.2a Binary Intrinsic (True/False)
`llm_eval_intrinsic_binary`: binary classification using `completion.md.jinja` (binary=True, extrinsic=False)


In [16]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


 Block 2: call the function with the code and the specifications from the data set

In [18]:
# llm_intrinsic_results = []
# eval_set_path = Path(EVAL_SET_DIR)
# for cve_dir in sorted(eval_set_path.iterdir()):
#     if not cve_dir.is_dir():
#         continue
#     cve_id = cve_dir.name
#     description = cve_descriptions.get(cve_id)
#     if description is None:
#         print(f"SKIP {cve_id}: no description in target_pool.json")
#         continue
#     for config_dir in sorted(cve_dir.iterdir()):
#         if not config_dir.is_dir():
#             continue
#         domain_file = config_dir / DOMAIN_FILE
#         problem_file = config_dir / PROBLEM_FILE
#         if not domain_file.exists():
#             continue
#         domain = domain_file.read_text(encoding='utf-8').strip()
#         problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
#         response = llm_eval_intrinsic(description, domain, problem, llm)
#         llm_intrinsic_results.append({
#             'cve_id': cve_id,
#             'config': config_dir.name,
#             'llm_response': response,
#         })
#         print(f"{cve_id}/{config_dir.name}  response: {response}")

In [17]:
# Preview: binary intrinsic prompt rendering
from ipywidgets import interact, IntSlider

def preview_intrinsic_binary(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'intrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=True, extrinsic=False,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )

    prompt = eval_template.render(**render_args)
    print(f'--- binary intrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_intrinsic_binary, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Because waiting more than 200ms didn’t work, add the free Qwen model to quickly validate the pipeline.

In [18]:

nvidia = OpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    timeout=120.0,
)

t_start_llm_bin = time.time()
total_tokens_bin = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

# Setup incremental save
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f"results_binary_{NVIDIA_MODEL.replace('/', '_')}.json")

# ── LLM intrinsic binary on reference data (incremental save) ──
llm_intrinsic_binary_results = []
for entry in dataset:
    for ap in entry["attack_paths"]:
        response, usage = llm_eval_intrinsic_binary(entry["cve_id"], entry["description"], ap["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
        for k in total_tokens_bin: total_tokens_bin[k] += usage.get(k, 0)
        
        # Parse label
        try:
            text = response.strip()
            if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
            label = json.loads(text).get("label", "?")
        except Exception:
            label = "?"
        
        result = {"cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "label": label, "llm_response": response, "usage": usage}
        llm_intrinsic_binary_results.append(result)
        
        # Incremental save after each domain
        with open(results_path, "w") as f:
            json.dump({"model": NVIDIA_MODEL, "results": llm_intrinsic_binary_results}, f, indent=2)
        
        print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {label} | {usage.get('elapsed_seconds', '')}s")

t_elapsed_llm_bin = time.time() - t_start_llm_bin

# Save final metadata
metadata_path = os.path.join(save_dir, f"metadata_binary_{NVIDIA_MODEL.replace('/', '_')}.json")
with open(metadata_path, "w") as f:
    json.dump({
        "time_seconds": round(t_elapsed_llm_bin, 2),
        "model": NVIDIA_MODEL,
        "n_domains": len(llm_intrinsic_binary_results),
        "total_tokens": total_tokens_bin,
        "device_info": "API (nvidia)",
    }, f, indent=2)

print(f"\nDone: {len(llm_intrinsic_binary_results)} domains")


CVE-2022-1471/AP1                            | True | 1.97s
CVE-2022-40149/AP1                            | True | 0.83s
CVE-2022-40149/AP2                            | True | 0.86s


KeyboardInterrupt: 

##### 3.3.1.2b Scored Intrinsic (12-criteria, integer 0-5)
`llm_eval_intrinsic_scored`: 12-criteria scored evaluation using `completion.md.jinja` (binary=False, extrinsic=False)


In [20]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def preview_intrinsic_scored(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'intrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=False, extrinsic=False,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )

    prompt = eval_template.render(**render_args)
    print(f'--- scored intrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_intrinsic_scored, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Block 2: evaluation result parser and formatter

In [23]:
SCORED_CRITERIA = ["F1","F2","A1","A2","C1","C2","C3","V1","V2","V3","N1","N2"]

def parse_scored_response(response_text):
    """Parse LLM scored response JSON, extract scores."""
    try:
        text = response_text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1]
            text = text.rsplit("```", 1)[0]
        data = json.loads(text)
        return data
    except Exception:
        return None

def domain_min_score(scores, criteria=SCORED_CRITERIA):
    """Return min of valid criteria scores (0-5). None if no valid scores."""
    valid = [scores[k] for k in criteria if isinstance(scores.get(k), (int, float))]
    return min(valid) if valid else None

def summarize_intrinsic_scored(result):
    """Extract one-line summary from intrinsic scored result."""
    data = parse_scored_response(result["llm_response"])
    if data is None:
        return f"{result['cve_id']}/{result.get('ap_id', result.get('config', '?'))}  PARSE_ERROR"
    qmin = domain_min_score(data)
    scores_str = " | ".join(str(data.get(k, "?")) for k in SCORED_CRITERIA)
    ap = result.get("ap_id", result.get("config", "?"))
    return f"{result['cve_id']}/{ap:30s} | {scores_str} | min={qmin}"

def summarize_extrinsic_scored(result):
    """Extract one-line summary from extrinsic scored result (+ R1-R4)."""
    data = parse_scored_response(result["llm_response"])
    if data is None:
        return f"{result['cve_id']}/{result.get('generated', '?')} vs {result.get('reference', '?')}  PARSE_ERROR"
    ext_keys = SCORED_CRITERIA + ["R1","R2","R3","R4"]
    qmin = domain_min_score(data)
    scores_str = " | ".join(str(data.get(k, "?")) for k in ext_keys)
    gen = result.get("generated", result.get("ap_id", "?"))
    ref = result.get("reference", "?")
    return f"{result['cve_id']}/{gen:20s} vs {ref} | {scores_str} | min={qmin}"


Block 3: call the function with the code and the specifications from the data set

In [24]:
t_start_llm_scored = time.time()
total_tokens_scored = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f"results_scored_{NVIDIA_MODEL.replace('/', '_')}.json")

# ── LLM intrinsic scored on reference data (incremental save) ──
llm_intrinsic_scored_results = []
for entry in dataset:
    for ap in entry["attack_paths"]:
        response, usage = llm_eval_intrinsic_scored(entry["cve_id"], entry["description"], ap["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
        for k in total_tokens_scored: total_tokens_scored[k] += usage.get(k, 0)

        scores = parse_scored_response(response) or {"parse_error": True}
        qmin = domain_min_score(scores) if "parse_error" not in scores else None

        result = {"cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "scores": scores, "min_score": qmin, "llm_response": response, "usage": usage}
        llm_intrinsic_scored_results.append(result)

        with open(results_path, "w") as f:
            json.dump({"model": NVIDIA_MODEL, "results": llm_intrinsic_scored_results}, f, indent=2)

        score_str = " ".join(f"{k}={scores.get(k, '?')}" for k in SCORED_CRITERIA)
        print(f"{entry['cve_id']}/{ap['ap_id']:30s}| score={qmin} | {usage.get('elapsed_seconds', '')}s")

t_elapsed_llm_scored = time.time() - t_start_llm_scored

metadata_path = os.path.join(save_dir, f"metadata_scored_{NVIDIA_MODEL.replace('/', '_')}.json")
with open(metadata_path, "w") as f:
    json.dump({
        "time_seconds": round(t_elapsed_llm_scored, 2),
        "model": NVIDIA_MODEL,
        "n_domains": len(llm_intrinsic_scored_results),
        "total_tokens": total_tokens_scored,
        "device_info": "API (nvidia)",
    }, f, indent=2)

print(f"\nDone: {len(llm_intrinsic_scored_results)} domains, {t_elapsed_llm_scored:.2f}s, Tokens: {total_tokens_scored}")


CVE-2022-1471/AP1                           | score=5 | 2.37s
CVE-2022-40149/AP1                           | score=5 | 3.14s


KeyboardInterrupt: 

In [81]:
load_dotenv(override=True)  # override=True Force overwrite existing environment variables.  
key = os.getenv("OPENAI_API_KEY", "")                                                                                   
print(f"Key loaded: {key[:8]}...{key[-4:]}" if len(key) > 12 else "Key NOT found or too short") 

Key loaded: sk-proj-...z4wA


In [82]:
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=120.0,
)

t_start_llm_gpt = time.time()
total_tokens_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f"results_scored_{GPT_MODEL.replace('.', '_')}.json")

# ── LLM intrinsic scored on reference data (GPT, incremental save) ──
llm_intrinsic_scored_results_gpt = []
for entry in dataset:
    for ap in entry["attack_paths"]:
        response, usage = llm_eval_intrinsic_scored(entry["cve_id"], entry["description"], ap["domain"], openai_client, GPT_MODEL, seed=SEED)
        for k in total_tokens_gpt: total_tokens_gpt[k] += usage.get(k, 0)

        scores = parse_scored_response(response) or {"parse_error": True}
        qmin = domain_min_score(scores) if "parse_error" not in scores else None

        result = {"cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "scores": scores, "min_score": qmin, "llm_response": response, "usage": usage}
        llm_intrinsic_scored_results_gpt.append(result)

        with open(results_path, "w") as f:
            json.dump({"model": GPT_MODEL, "results": llm_intrinsic_scored_results_gpt}, f, indent=2)

        score_str = " ".join(f"{k}={scores.get(k, '?')}" for k in SCORED_CRITERIA)
        print(f"{entry['cve_id']}/{ap['ap_id']:30s} | score={qmin} | {usage.get('elapsed_seconds', '')}s")

t_elapsed_llm_gpt = time.time() - t_start_llm_gpt

metadata_path = os.path.join(save_dir, f"metadata_scored_{GPT_MODEL.replace('.', '_')}.json")
with open(metadata_path, "w") as f:
    json.dump({
        "time_seconds": round(t_elapsed_llm_gpt, 2),
        "model": GPT_MODEL,
        "n_domains": len(llm_intrinsic_scored_results_gpt),
        "total_tokens": total_tokens_gpt,
        "device_info": "API (OpenAI)",
    }, f, indent=2)

print(f"\nDone: {len(llm_intrinsic_scored_results_gpt)} domains, {t_elapsed_llm_gpt:.2f}s, Tokens: {total_tokens_gpt}")


CVE-2022-1471/AP1                            | score=5 | 2.69s
CVE-2022-40149/AP1                            | score=5 | 2.23s


KeyboardInterrupt: 

#### 3.3.2 Extrinsic

##### 3.3.2.1a Embedding: reference PDDL vs generated PDDL Similarity

Block 1: the function to compute the embedding similarity between two blocks of PDDL code (domain + problem)

In [25]:
def embedding_similarity_extrinsic(pddl_texts_generated, pddl_texts_reference, model):
    """Cosine similarity matrix between generated and reference PDDL codes.
    Args:
        pddl_texts_generated: list of generated PDDL domain strings
        pddl_texts_reference: list of reference PDDL domain strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(generated), len(reference))
    """
    E_generated = model.encode(
        pddl_texts_generated,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_reference = model.encode(
        pddl_texts_reference,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_generated @ E_reference.T
    return S.float().cpu().numpy()

Block 2: Extrinsic cross-validation, performance estimation, and corrected proportion
Same process as intrinsic Block 5 but for extrinsic (domain vs domain):
1. Build positive/negative pairs: 
  - 1：CVE PDDL AP vs the PDDL APs of the same CVE 
  - 0：CVE PDDL AP vs the PDDL APs of the different CVE  
2. CVE-level GroupKFold CV with bootstrap threshold
3. Out-of-fold predictions → classification report → TPR, FPR
4. Apply threshold to reference domains

In [26]:
# ── Step 1: Build extrinsic positive/negative pairs ──

def build_extrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for extrinsic embedding evaluation.
    Positive (1): same CVE, different APs (domain_i vs domain_j)
    Negative (0): different CVE (domain_i vs domain_j)
    Groups: assigned by CVE of first domain in pair
    """
    # Collect all (domain, cve_id) pairs
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))

    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]

    # Batch encode
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()

    scores, labels, groups = [], [], []
    n = len(all_entries)
    for i in range(n):
        for j in range(i+1, n):  # upper triangle only, symmetric
            sim = float(sim_matrix[i, j])
            label = 1 if cve_ids[i] == cve_ids[j] else 0
            scores.append(sim)
            labels.append(label)
            groups.append(cve_ids[i])  # group by first domain's CVE

    return np.array(scores), np.array(labels), np.array(groups)


ext_scores_a, ext_labels_a, ext_groups_a = build_extrinsic_pairs(dataset, embedding_model)
print(f"Extrinsic Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {ext_labels_a.sum()}, Negative pairs: {(ext_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(ext_groups_a))}")


Extrinsic Embedding Model: all-MiniLM-L6-v2
  Positive pairs: 74, Negative pairs: 1411
  Groups (CVEs): 21


In [27]:
# ── Step 2: Extrinsic CV calibration ──

ext_cv_threshold_a, ext_cv_fold_thr_a, ext_cv_pred_a, ext_cv_true_a = run_calibration(
    ext_scores_a, ext_labels_a, ext_groups_a)
print(f"\nMedian threshold: {ext_cv_threshold_a:.4f}")
print("Fold thresholds:", [f"{t:.4f}" for t in ext_cv_fold_thr_a])


  Fold 1: threshold=0.9621, val CVEs=['CVE-2022-1471', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-24813']
  Fold 2: threshold=0.9744, val CVEs=['CVE-2023-44487', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2024-47072']
  Fold 3: threshold=0.9383, val CVEs=['CVE-2022-40150', 'CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-38816']
  Fold 4: threshold=0.9621, val CVEs=['CVE-2024-22243', 'CVE-2024-22259', 'CVE-2024-34447', 'CVE-2024-38286']
  Fold 5: threshold=0.9621, val CVEs=['CVE-2022-40149', 'CVE-2023-33202', 'CVE-2023-34055', 'CVE-2025-22228']

Median threshold: 0.9621
Fold thresholds: ['0.9621', '0.9744', '0.9383', '0.9621', '0.9621']


In [28]:
# ── Step 3: Extrinsic performance report ──

print(f"Extrinsic Embedding Model ({EMBEDDING_MODEL_NAME})")
print(classification_report(ext_cv_true_a, ext_cv_pred_a, zero_division=0))

ext_row_a = report_row(ext_cv_true_a, ext_cv_pred_a,
                      metric="embedding", model=EMBEDDING_MODEL_NAME,
                      mode="extrinsic", threshold=ext_cv_threshold_a)
print(f"TPR = {ext_row_a['tpr']:.4f}  FPR = {ext_row_a['fpr']:.4f}")


Extrinsic Embedding Model (all-MiniLM-L6-v2)
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1411
           1       0.45      0.92      0.61        74

    accuracy                           0.94      1485
   macro avg       0.72      0.93      0.79      1485
weighted avg       0.97      0.94      0.95      1485

TPR = 0.9189  FPR = 0.0581


In [29]:
# ── Step 4: Apply extrinsic threshold to reference domains ──
# For extrinsic, apply threshold to all intra-CVE AP pairs
ext_ref_results_a = []
for entry in dataset:
    ref_aps = entry["attack_paths"]
    if len(ref_aps) < 2:
        continue
    texts = [ap["domain"] for ap in ref_aps]
    E = embedding_model.encode(texts, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    S = (E @ E.T).float().cpu().numpy()
    for i in range(len(ref_aps)):
        for j in range(i+1, len(ref_aps)):
            sim = float(S[i, j])
            pred = 1 if sim >= ext_cv_threshold_a else 0
            label = f"{entry['cve_id']}/{ref_aps[i]['ap_id']} vs {ref_aps[j]['ap_id']}"
            ext_ref_results_a.append({"label": label, "similarity": sim, "prediction": pred})
            print(f"  {label:45s} prediction: {pred} (similarity: {sim:.4f})")

ext_ref_ppv_a = sum(r["prediction"] for r in ext_ref_results_a) / len(ext_ref_results_a) if ext_ref_results_a else 0
print(f"  Extrinsic Reference PPV: {ext_ref_ppv_a:.4f} ({sum(r['prediction'] for r in ext_ref_results_a)}/{len(ext_ref_results_a)} predicted positive)")

# Save
ext_save_data = {
    "metric": "embedding", "model": EMBEDDING_MODEL_NAME, "mode": "extrinsic",
    "median_threshold": ext_cv_threshold_a, "fold_thresholds": ext_cv_fold_thr_a,
    "tpr": ext_row_a["tpr"], "fpr": ext_row_a["fpr"],
    "classification_report": classification_report(ext_cv_true_a, ext_cv_pred_a, output_dict=True, zero_division=0),
    "n_positive_pairs": int(ext_labels_a.sum()), "n_negative_pairs": int((ext_labels_a == 0).sum()),
    "ref_ppv": ext_ref_ppv_a, "reference_predictions": ext_ref_results_a,
}
ext_save_path = Path(EVAL_SET_DIR) / f"cv_calibration_extrinsic_{EMBEDDING_MODEL_NAME.replace('/', '_')}.json"
ext_save_path.write_text(json.dumps(ext_save_data, indent=2), encoding="utf-8")

calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME, "mode": "extrinsic", "threshold": ext_cv_threshold_a,
    "tpr": ext_row_a["tpr"], "fpr": ext_row_a["fpr"],
    "1__precision": ext_row_a.get("1__precision", float("nan")),
    "1__recall": ext_row_a.get("1__recall", float("nan")),
    "1__f1-score": ext_row_a.get("1__f1-score", float("nan")),
    "accuracy": ext_row_a.get("accuracy", float("nan")),
    "ref_ppv": ext_ref_ppv_a,
    "n_positive": int(ext_labels_a.sum()), "n_negative": int((ext_labels_a == 0).sum()),
})


  CVE-2022-40149/AP1 vs AP2                     prediction: 1 (similarity: 1.0000)
  CVE-2022-40150/AP1 vs AP2                     prediction: 1 (similarity: 1.0000)
  CVE-2023-2976/AP1 vs AP2                      prediction: 1 (similarity: 0.9975)
  CVE-2023-2976/AP1 vs AP3                      prediction: 1 (similarity: 0.9877)
  CVE-2023-2976/AP2 vs AP3                      prediction: 1 (similarity: 0.9881)
  CVE-2023-33202/AP1 vs AP2                     prediction: 1 (similarity: 0.9853)
  CVE-2023-33202/AP1 vs AP3                     prediction: 1 (similarity: 0.9770)
  CVE-2023-33202/AP2 vs AP3                     prediction: 1 (similarity: 0.9762)
  CVE-2023-46589/AP1 vs AP2                     prediction: 1 (similarity: 0.9704)
  CVE-2023-46589/AP1 vs AP3                     prediction: 1 (similarity: 1.0000)
  CVE-2023-46589/AP1 vs AP4                     prediction: 1 (similarity: 0.9704)
  CVE-2023-46589/AP2 vs AP3                     prediction: 1 (similarity: 0.9704)
  CV

##### 3.3.2.2 LLM as an Expert — Reference PDDL vs Candidate PDDL Match

##### 3.3.2.2a Binary Extrinsic (True/False)
`llm_eval_extrinsic_binary`: binary classification using `completion.md.jinja` (binary=True, extrinsic=True)


In [30]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


Block 2: call the function with the code and the specifications from the data set

In [31]:
def preview_extrinsic_binary(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'extrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=True, extrinsic=True,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )
    ref = few_shot_pool[1] if len(few_shot_pool) > 1 else few_shot_pool[0]
    render_args['domain_reference'] = ref.domain_pddl
    prompt = eval_template.render(**render_args)
    print(f'--- binary extrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_extrinsic_binary, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

In [32]:
t_start_ext_bin = time.time()
total_tokens_ext_bin = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f"results_binary_{NVIDIA_MODEL.replace('/', '_')}.json")

# ── LLM extrinsic binary on reference data (incremental save) ──
llm_extrinsic_binary_results = []
for entry in dataset:
    ref_aps = entry["attack_paths"]
    if len(ref_aps) < 2:
        continue
    for i, ap_gen in enumerate(ref_aps):
        for j, ap_ref in enumerate(ref_aps):
            # same-AP (i==j) included: tests self-alignment
            response, usage = llm_eval_extrinsic_binary(
                entry["cve_id"], entry["description"], ap_ref["domain"],
                ap_gen["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
            for k in total_tokens_ext_bin: total_tokens_ext_bin[k] += usage.get(k, 0)
            
            try:
                text = response.strip()
                if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
                label = json.loads(text).get("label", "?")
            except Exception:
                label = "?"
            
            result = {
                "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
                "reference": ap_ref["ap_id"], "label": label, "llm_response": response, "usage": usage,
            }
            llm_extrinsic_binary_results.append(result)
            
            with open(results_path, "w") as f:
                json.dump({"model": NVIDIA_MODEL, "results": llm_extrinsic_binary_results}, f, indent=2)
            
            print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s")

t_elapsed_ext_bin = time.time() - t_start_ext_bin

metadata_path = os.path.join(save_dir, f"metadata_binary_{NVIDIA_MODEL.replace('/', '_')}.json")
with open(metadata_path, "w") as f:
    json.dump({
        "time_seconds": round(t_elapsed_ext_bin, 2),
        "model": NVIDIA_MODEL,
        "n_pairs": len(llm_extrinsic_binary_results),
        "total_tokens": total_tokens_ext_bin,
        "device_info": "API (nvidia)",
    }, f, indent=2)

print(f"\nDone: {len(llm_extrinsic_binary_results)} pairs, {t_elapsed_ext_bin:.2f}s, Tokens: {total_tokens_ext_bin}")


CVE-2022-40149/AP1                  vs AP1 | True | 1.34s
CVE-2022-40149/AP1                  vs AP2 | True | 3.06s
CVE-2022-40149/AP2                  vs AP1 | False | 1.63s


KeyboardInterrupt: 

##### 3.3.2.2b Scored Extrinsic (16-criteria, integer 0-5)
`llm_eval_extrinsic_scored`: 16-criteria scored evaluation (12 quality + 4 alignment) using `completion.md.jinja` (binary=False, extrinsic=True)


In [33]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [34]:
def preview_extrinsic_scored(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'extrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=False, extrinsic=True,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )
    ref = few_shot_pool[1] if len(few_shot_pool) > 1 else few_shot_pool[0]
    render_args['domain_reference'] = ref.domain_pddl
    prompt = eval_template.render(**render_args)
    print(f'--- scored extrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_extrinsic_scored, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Block 2: call the function with the code and the specifications from the data set

In [44]:
t_start_ext_scored = time.time()
total_tokens_ext_scored = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f"results_scored_{NVIDIA_MODEL.replace('/', '_')}.json")

EXT_SCORED_CRITERIA = SCORED_CRITERIA + ["R1","R2","R3","R4"]

# ── LLM extrinsic scored on reference data (incremental save) ──
llm_extrinsic_scored_results = []
for entry in dataset:
    ref_aps = entry["attack_paths"]
    if len(ref_aps) < 2:
        continue
    for i, ap_gen in enumerate(ref_aps):
        for j, ap_ref in enumerate(ref_aps):
            # same-AP (i==j) included: tests self-alignment
            response, usage = llm_eval_extrinsic_scored(
                entry["cve_id"], entry["description"], ap_ref["domain"],
                ap_gen["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
            for k in total_tokens_ext_scored: total_tokens_ext_scored[k] += usage.get(k, 0)

            scores = parse_scored_response(response) or {"parse_error": True}
            qmin = domain_min_score(scores) if "parse_error" not in scores else None

            result = {
                "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
                "reference": ap_ref["ap_id"], "scores": scores, "min_score": qmin,
                "llm_response": response, "usage": usage,
            }
            llm_extrinsic_scored_results.append(result)

            with open(results_path, "w") as f:
                json.dump({"model": NVIDIA_MODEL, "results": llm_extrinsic_scored_results}, f, indent=2)

            score_str = " ".join(f"{k}={scores.get(k, '?')}" for k in EXT_SCORED_CRITERIA)
            print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | score={qmin} | {usage.get('elapsed_seconds', '')}s")

t_elapsed_ext_scored = time.time() - t_start_ext_scored

metadata_path = os.path.join(save_dir, f"metadata_scored_{NVIDIA_MODEL.replace('/', '_')}.json")
with open(metadata_path, "w") as f:
    json.dump({
        "time_seconds": round(t_elapsed_ext_scored, 2),
        "model": NVIDIA_MODEL,
        "n_pairs": len(llm_extrinsic_scored_results),
        "total_tokens": total_tokens_ext_scored,
        "device_info": "API (nvidia)",
    }, f, indent=2)

print(f"\nDone: {len(llm_extrinsic_scored_results)} pairs, {t_elapsed_ext_scored:.2f}s, Tokens: {total_tokens_ext_scored}")


CVE-2022-40149/AP1                  vs AP1 | score=5 | 3.36s
CVE-2022-40149/AP1                  vs AP2 | score=5 | 3.27s


KeyboardInterrupt: 

Block 3: call the function with GPT-4.1-mini (reuses `parse_scored_response` and `summarize_extrinsic_scored` from intrinsic section)

In [45]:
t_start_ext_gpt = time.time()
total_tokens_ext_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f"results_scored_{GPT_MODEL.replace('.', '_')}.json")

# ── LLM extrinsic scored on reference data (GPT, incremental save) ──
llm_extrinsic_scored_results_gpt = []
for entry in dataset:
    ref_aps = entry["attack_paths"]
    if len(ref_aps) < 2:
        continue
    for i, ap_gen in enumerate(ref_aps):
        for j, ap_ref in enumerate(ref_aps):
            if i == j:
                continue
            response, usage = llm_eval_extrinsic_scored(
                entry["cve_id"], entry["description"], ap_ref["domain"],
                ap_gen["domain"], openai_client, GPT_MODEL, seed=SEED)
            for k in total_tokens_ext_gpt: total_tokens_ext_gpt[k] += usage.get(k, 0)

            scores = parse_scored_response(response) or {"parse_error": True}
            qmin = domain_min_score(scores) if "parse_error" not in scores else None

            result = {
                "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
                "reference": ap_ref["ap_id"], "scores": scores, "min_score": qmin,
                "llm_response": response, "usage": usage,
            }
            llm_extrinsic_scored_results_gpt.append(result)

            with open(results_path, "w") as f:
                json.dump({"model": GPT_MODEL, "results": llm_extrinsic_scored_results_gpt}, f, indent=2)

            score_str = " ".join(f"{k}={scores.get(k, '?')}" for k in EXT_SCORED_CRITERIA)
            print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | score={qmin} | {usage.get('elapsed_seconds', '')}s")

t_elapsed_ext_gpt = time.time() - t_start_ext_gpt

metadata_path = os.path.join(save_dir, f"metadata_scored_{GPT_MODEL.replace('.', '_')}.json")
with open(metadata_path, "w") as f:
    json.dump({
        "time_seconds": round(t_elapsed_ext_gpt, 2),
        "model": GPT_MODEL,
        "n_pairs": len(llm_extrinsic_scored_results_gpt),
        "total_tokens": total_tokens_ext_gpt,
        "device_info": "API (OpenAI)",
    }, f, indent=2)

print(f"\nDone: {len(llm_extrinsic_scored_results_gpt)} pairs, {t_elapsed_ext_gpt:.2f}s, Tokens: {total_tokens_ext_gpt}")


CVE-2022-40149/AP1                  vs AP2 | score=5 | 3.1s


KeyboardInterrupt: 